# Tablero de Mando - Análisis Vintage de Mora Crediticia CA

Este notebook ejecuta el pipeline completo de análisis vintage y presenta los resultados clave.

**Pasos:**
1. Consolidar archivos de otorgamientos
2. Generar matriz vintage
3. Visualizar curvas vintage
4. Resumen ejecutivo

---
## 0. Setup

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D

# Configurar rutas
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# ---------------------------------------------------------------------------
# PARÁMETROS CONFIGURABLES (modificar acá)
# ---------------------------------------------------------------------------
N_DESTACAR = 5                  # Cantidad de últimas cohortes a resaltar
COLOR_2024 = "#2171b5"          # Color para cohortes 2024 (azul)
COLOR_2025 = "#e6550d"          # Color para cohortes 2025 (naranja)
ALPHA_INDIVIDUAL = 0.18         # Transparencia de curvas individuales
ALPHA_HIGHLIGHT = 0.85          # Transparencia de las curvas destacadas

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Cohortes a resaltar: últimas {N_DESTACAR}")

---
## 1. Consolidar archivos de otorgamientos

In [ ]:
result = subprocess.run(
    ["py", os.path.join(SRC_DIR, "consolidar_vintage.py")],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
df = pd.read_csv(os.path.join(PROCESSED_DIR, "vintage_consolidado.csv"), sep=";", decimal=",")
print(f"Registros: {len(df)} | Cohortes: {df['cohorte'].nunique()} | MOB máximo: {df['mob'].max()}")
df.head(10)

---
## 2. Generar matriz vintage

In [ ]:
result = subprocess.run(
    ["py", os.path.join(SRC_DIR, "generar_matriz_vintage.py")],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
matriz = pd.read_csv(os.path.join(PROCESSED_DIR, "matriz_vintage.csv"), sep=";", decimal=",", index_col=0)

# Mostrar con formato porcentaje
matriz.style.format("{:.2%}", na_rep="-").background_gradient(cmap="RdYlGn_r", axis=None)

---
## 3. Curvas Vintage

In [ ]:
# Usa N_DESTACAR, COLOR_2024, COLOR_2025, ALPHA_INDIVIDUAL, ALPHA_HIGHLIGHT del Setup
LW_INDIVIDUAL = 1.0
LW_PROMEDIO = 2.8
LW_HIGHLIGHT = 2.2

# Paleta de colores distinguibles para las últimas N cohortes destacadas
COLORES_DESTACADAS = ["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"]

cohortes_ordenadas = sorted(matriz.index)
ultimas_n = cohortes_ordenadas[-N_DESTACAR:]
color_por_cohorte = {c: COLORES_DESTACADAS[i % len(COLORES_DESTACADAS)] for i, c in enumerate(ultimas_n)}

cohortes_2024 = [c for c in cohortes_ordenadas if c.startswith("2024")]
cohortes_2025 = [c for c in cohortes_ordenadas if c.startswith("2025")]
cohortes_2026 = [c for c in cohortes_ordenadas if c.startswith("2026")]

def extraer_serie(cohorte):
    valores = matriz.loc[cohorte].dropna()
    mobs = [int(c.replace("MOB_", "")) for c in valores.index]
    return mobs, valores.values

def promedio_anio(cohortes):
    mob_max = max(int(c.replace("MOB_", "")) for c in matriz.columns)
    mobs_out, vals_out = [], []
    for m in range(1, mob_max + 1):
        col = f"MOB_{m}"
        vals = matriz.loc[cohortes, col].dropna()
        if len(vals) >= 2:
            mobs_out.append(m)
            vals_out.append(vals.mean())
    return mobs_out, vals_out

prom_mobs_2024, prom_vals_2024 = promedio_anio(cohortes_2024)
prom_mobs_2025, prom_vals_2025 = promedio_anio(cohortes_2025) if len(cohortes_2025) >= 2 else ([], [])

fig, ax = plt.subplots(figsize=(15, 7))

for cohorte in cohortes_2024:
    mobs, vals = extraer_serie(cohorte)
    es_destacada = cohorte in ultimas_n
    ax.plot(mobs, vals,
            color=color_por_cohorte[cohorte] if es_destacada else COLOR_2024,
            alpha=ALPHA_HIGHLIGHT if es_destacada else ALPHA_INDIVIDUAL,
            linewidth=LW_HIGHLIGHT if es_destacada else LW_INDIVIDUAL,
            zorder=3 if es_destacada else 1)

for cohorte in cohortes_2025:
    mobs, vals = extraer_serie(cohorte)
    es_destacada = cohorte in ultimas_n
    ax.plot(mobs, vals,
            color=color_por_cohorte[cohorte] if es_destacada else COLOR_2025,
            alpha=ALPHA_HIGHLIGHT if es_destacada else ALPHA_INDIVIDUAL,
            linewidth=LW_HIGHLIGHT if es_destacada else LW_INDIVIDUAL,
            zorder=3 if es_destacada else 1)

for cohorte in cohortes_2026:
    mobs, vals = extraer_serie(cohorte)
    es_destacada = cohorte in ultimas_n
    ax.plot(mobs, vals,
            color=color_por_cohorte[cohorte] if es_destacada else "gray",
            alpha=ALPHA_HIGHLIGHT if es_destacada else ALPHA_INDIVIDUAL,
            linewidth=LW_HIGHLIGHT if es_destacada else LW_INDIVIDUAL,
            zorder=3 if es_destacada else 1)

if prom_mobs_2024:
    ax.plot(prom_mobs_2024, prom_vals_2024,
            color=COLOR_2024, linewidth=LW_PROMEDIO, linestyle="--",
            marker="s", markersize=5, zorder=4)

if prom_mobs_2025:
    ax.plot(prom_mobs_2025, prom_vals_2025,
            color=COLOR_2025, linewidth=LW_PROMEDIO, linestyle="--",
            marker="s", markersize=5, zorder=4)

for cohorte in ultimas_n:
    mobs, vals = extraer_serie(cohorte)
    color = color_por_cohorte[cohorte]
    ax.annotate(cohorte, xy=(mobs[-1], vals[-1]),
                xytext=(6, 0), textcoords="offset points",
                fontsize=9, fontweight="bold", color=color, va="center",
                zorder=5)
    ax.plot(mobs[-1], vals[-1], "o", color=color, markersize=6, zorder=5)

ax.set_xlabel("Months on Books (MOB)", fontsize=12)
ax.set_ylabel("Índice de Morosidad", fontsize=12)
ax.set_title("Curvas Vintage — Índice de Morosidad por Cohorte", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xticks(range(1, int(df["mob"].max()) + 1))
ax.grid(True, alpha=0.25, linestyle="--")
ax.set_xlim(0.5, int(df["mob"].max()) + 1.5)

legend_elements = [
    Line2D([0], [0], color=COLOR_2024, alpha=ALPHA_INDIVIDUAL, lw=4, label="Cohortes 2024 (individual)"),
    Line2D([0], [0], color=COLOR_2024, lw=LW_PROMEDIO, ls="--", marker="s", markersize=5, label="Promedio 2024"),
    Line2D([0], [0], color=COLOR_2025, alpha=ALPHA_INDIVIDUAL, lw=4, label="Cohortes 2025 (individual)"),
]
if prom_mobs_2025:
    legend_elements.append(
        Line2D([0], [0], color=COLOR_2025, lw=LW_PROMEDIO, ls="--", marker="s", markersize=5, label="Promedio 2025"))
for cohorte in ultimas_n:
    legend_elements.append(
        Line2D([0], [0], color=color_por_cohorte[cohorte], lw=LW_HIGHLIGHT, alpha=ALPHA_HIGHLIGHT, label=cohorte))

ax.legend(handles=legend_elements, loc="upper right", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "reports", "curvas_vintage.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado en reports/curvas_vintage.png")

---
## 3b. Factores de Desarrollo

El factor de desarrollo mide cuánto cambia el índice de morosidad de un MOB al siguiente:
**factor = índice(MOB_n) / índice(MOB_n-1)**

- **Factor > 1**: la mora crece → el deterioro se acelera
- **Factor = 1**: la mora se mantiene estable
- **Factor < 1**: la mora se reduce → la cartera se estabiliza o recupera

In [ ]:

# Ejecutar script de factores de desarrollo
result = subprocess.run(
    ["py", os.path.join(SRC_DIR, "generar_factores_desarrollo.py")],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)


In [ ]:

# Leer factores y mostrar como heatmap con formato condicional
factores = pd.read_csv(os.path.join(PROCESSED_DIR, "factores_desarrollo.csv"), sep=";", decimal=",", index_col=0)

# Colorear: rojo si > 1 (mora crece), verde si < 1 (mora baja), blanco en 1
factores.style.format("{:.3f}", na_rep="-").background_gradient(
    cmap="RdYlGn_r", vmin=0.90, vmax=1.30, axis=None
)


### Curvas de factores de desarrollo por transición MOB

Cada línea muestra cómo evoluciona el factor de desarrollo (mes a mes) para una cohorte.
La **línea roja punteada en 1.0** es el umbral: por encima la mora crece, por debajo se estabiliza.

Si todas las cohortes cruzan el umbral en un MOB similar, el patrón es **estructural**.
Si hay cohortes que divergen, hay un **cambio de tendencia** en la calidad crediticia.

In [ ]:
from matplotlib.lines import Line2D
import numpy as np

COLOR_2024 = "#2171b5"
COLOR_2025 = "#e6550d"
ALPHA_IND = 0.20
ALPHA_HL = 0.85
N_DESTACAR = 3

cohortes_ord = sorted(factores.index)
ultimas_n = cohortes_ord[-N_DESTACAR:]
cohortes_2024 = [c for c in cohortes_ord if c.startswith("2024")]
cohortes_2025 = [c for c in cohortes_ord if c.startswith("2025")]

def extraer_factores(cohorte):
    vals = factores.loc[cohorte].dropna()
    if len(vals) == 0:
        return [], np.array([])
    mobs = [int(t.split("->")[1]) for t in vals.index]
    return mobs, vals.values

def promedio_factores(cohortes):
    mobs_out, vals_out = [], []
    for col in factores.columns:
        vals = factores.loc[cohortes, col].dropna()
        if len(vals) >= 2:
            mob_dest = int(col.split("->")[1])
            mobs_out.append(mob_dest)
            vals_out.append(vals.mean())
    return mobs_out, vals_out

prom_m24, prom_v24 = promedio_factores(cohortes_2024)
prom_m25, prom_v25 = promedio_factores(cohortes_2025) if len(cohortes_2025) >= 2 else ([], [])

fig, ax = plt.subplots(figsize=(15, 7))

ax.axhline(y=1.0, color="#d62728", linewidth=1.5, linestyle=":", alpha=0.7, zorder=0)
ax.axhspan(1.0, 1.35, alpha=0.04, color="red", zorder=0)
ax.axhspan(0.85, 1.0, alpha=0.04, color="green", zorder=0)

for cohorte in cohortes_2024:
    mobs, vals = extraer_factores(cohorte)
    if not mobs:
        continue
    hl = cohorte in ultimas_n
    ax.plot(mobs, vals, color=COLOR_2024,
            alpha=ALPHA_HL if hl else ALPHA_IND,
            linewidth=2.2 if hl else 1.0, zorder=3 if hl else 1)

for cohorte in cohortes_2025:
    mobs, vals = extraer_factores(cohorte)
    if not mobs:
        continue
    hl = cohorte in ultimas_n
    ax.plot(mobs, vals, color=COLOR_2025,
            alpha=ALPHA_HL if hl else ALPHA_IND,
            linewidth=2.2 if hl else 1.0, zorder=3 if hl else 1)

if prom_m24:
    ax.plot(prom_m24, prom_v24, color=COLOR_2024, linewidth=2.8,
            linestyle="--", marker="s", markersize=5, zorder=4)
if prom_m25:
    ax.plot(prom_m25, prom_v25, color=COLOR_2025, linewidth=2.8,
            linestyle="--", marker="s", markersize=5, zorder=4)

for cohorte in ultimas_n:
    mobs, vals = extraer_factores(cohorte)
    if not mobs:
        continue
    color = COLOR_2024 if cohorte.startswith("2024") else COLOR_2025
    ax.annotate(cohorte, xy=(mobs[-1], vals[-1]),
                xytext=(6, 0), textcoords="offset points",
                fontsize=9, fontweight="bold", color=color, va="center", zorder=5)
    ax.plot(mobs[-1], vals[-1], "o", color=color, markersize=6, zorder=5)

ax.text(0.98, 0.95, "MORA CRECE", transform=ax.transAxes, fontsize=9,
        ha="right", va="top", color="#d62728", alpha=0.5, fontstyle="italic")
ax.text(0.98, 0.05, "MORA SE ESTABILIZA", transform=ax.transAxes, fontsize=9,
        ha="right", va="bottom", color="#2ca02c", alpha=0.5, fontstyle="italic")

ax.set_xlabel("MOB destino", fontsize=12)
ax.set_ylabel("Factor de Desarrollo", fontsize=12)
ax.set_title("Factores de Desarrollo — Variación del Índice entre MOBs Consecutivos",
             fontsize=14, fontweight="bold")
mob_max = max(int(c.split("->")[1]) for c in factores.columns)
ax.set_xticks(range(2, mob_max + 1))
ax.grid(True, alpha=0.25, linestyle="--")
ax.set_xlim(1.5, mob_max + 1.5)

legend_elements = [
    Line2D([0], [0], color=COLOR_2024, alpha=ALPHA_IND, lw=4, label="Cohortes 2024"),
    Line2D([0], [0], color=COLOR_2024, lw=2.8, ls="--", marker="s", markersize=5, label="Promedio 2024"),
    Line2D([0], [0], color=COLOR_2025, alpha=ALPHA_IND, lw=4, label="Cohortes 2025"),
]
if prom_m25:
    legend_elements.append(
        Line2D([0], [0], color=COLOR_2025, lw=2.8, ls="--", marker="s", markersize=5, label="Promedio 2025"))
legend_elements.append(
    Line2D([0], [0], color="#d62728", lw=1.5, ls=":", label="Umbral (factor=1)"))

ax.legend(handles=legend_elements, loc="upper right", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "reports", "factores_desarrollo.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado en reports/factores_desarrollo.png")

In [ ]:
print("ANÁLISIS DE PUNTO DE QUIEBRE")
print("=" * 60)
print("MOB donde el factor promedio baja de 1.0 (mora empieza a estabilizarse):\n")

for year, cohs in [("2024", cohortes_2024), ("2025", cohortes_2025)]:
    if len(cohs) < 2:
        continue
    print(f"  Promedio {year}:")
    for col in factores.columns:
        vals = factores.loc[cohs, col].dropna()
        if len(vals) >= 2:
            mean_val = vals.mean()
            mob_dest = col.split("->")[1]
            marker = " ◄ CRUCE" if abs(mean_val - 1.0) < 0.015 else ""
            zona = "▲" if mean_val > 1.0 else "▼"
            print(f"    MOB {mob_dest:>2s}: {mean_val:.4f} {zona}{marker}")

print("\n" + "=" * 60)
print("\nInterpretación:")
print("  ▲ Factor > 1: la mora sigue creciendo en esa transición")
print("  ▼ Factor < 1: la mora empieza a bajar en esa transición")
print("  ◄ CRUCE: punto donde el factor está muy cerca de 1 (cambio de tendencia)")


---
## 3c. Velocidad de Mora (1ra derivada) y Diagnóstico de Fase

La **velocidad** mide el cambio absoluto del índice entre MOBs consecutivos:
`velocidad = índice(MOB_n) - índice(MOB_n-1)`

| Velocidad | Factor | Diagnóstico |
|-----------|--------|-------------|
| > 0 | > 1 y subiendo | **Aceleración**: deterioro se intensifica |
| > 0 | > 1 pero bajando hacia 1 | **Desaceleración**: mora crece pero cada vez más lento |
| ≈ 0 | ≈ 1 | **Plateau/Meseta**: la cosecha se estabilizó |
| < 0 | < 1 | **Curación**: la cosecha está mejorando |


In [ ]:
result = subprocess.run(
    ["py", os.path.join(SRC_DIR, "generar_velocidad_mora.py")],
    capture_output=True, text=True, cwd=PROJECT_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)


In [ ]:
velocidad = pd.read_csv(os.path.join(PROCESSED_DIR, "velocidad_mora.csv"), sep=";", decimal=",", index_col=0)

# Heatmap: rojo si > 0 (mora crece), verde si < 0 (mora baja), blanco en 0
velocidad.style.format("{:.4f}", na_rep="-").background_gradient(
    cmap="RdYlGn_r", vmin=-0.02, vmax=0.04, axis=None
)


### Curvas de velocidad por cohorte

La línea punteada en **0** marca el plateau. Por encima la mora crece, por debajo mejora.


In [ ]:
from matplotlib.lines import Line2D

COLOR_2024 = "#2171b5"
COLOR_2025 = "#e6550d"
ALPHA_IND = 0.20
ALPHA_HL = 0.85
N_DESTACAR = 3

cohortes_ord = sorted(velocidad.index)
ultimas_n = cohortes_ord[-N_DESTACAR:]
cohortes_2024 = [c for c in cohortes_ord if c.startswith("2024")]
cohortes_2025 = [c for c in cohortes_ord if c.startswith("2025")]

def extraer_vel(cohorte):
    vals = velocidad.loc[cohorte].dropna()
    if len(vals) == 0:
        return [], np.array([])
    mobs = [int(t.split("->")[1]) for t in vals.index]
    return mobs, vals.values

def promedio_vel(cohortes):
    mobs_out, vals_out = [], []
    for col in velocidad.columns:
        vals = velocidad.loc[cohortes, col].dropna()
        if len(vals) >= 2:
            mobs_out.append(int(col.split("->")[1]))
            vals_out.append(vals.mean())
    return mobs_out, vals_out

pv_m24, pv_v24 = promedio_vel(cohortes_2024)
pv_m25, pv_v25 = promedio_vel(cohortes_2025) if len(cohortes_2025) >= 2 else ([], [])

fig, ax = plt.subplots(figsize=(15, 7))

ax.axhline(y=0, color="#d62728", linewidth=1.5, linestyle=":", alpha=0.7, zorder=0)
ax.axhspan(0, 0.06, alpha=0.04, color="red", zorder=0)
ax.axhspan(-0.03, 0, alpha=0.04, color="green", zorder=0)

for cohorte in cohortes_2024:
    mobs, vals = extraer_vel(cohorte)
    if not mobs:
        continue
    hl = cohorte in ultimas_n
    ax.plot(mobs, vals, color=COLOR_2024,
            alpha=ALPHA_HL if hl else ALPHA_IND,
            linewidth=2.2 if hl else 1.0, zorder=3 if hl else 1)

for cohorte in cohortes_2025:
    mobs, vals = extraer_vel(cohorte)
    if not mobs:
        continue
    hl = cohorte in ultimas_n
    ax.plot(mobs, vals, color=COLOR_2025,
            alpha=ALPHA_HL if hl else ALPHA_IND,
            linewidth=2.2 if hl else 1.0, zorder=3 if hl else 1)

if pv_m24:
    ax.plot(pv_m24, pv_v24, color=COLOR_2024, linewidth=2.8,
            linestyle="--", marker="s", markersize=5, zorder=4)
if pv_m25:
    ax.plot(pv_m25, pv_v25, color=COLOR_2025, linewidth=2.8,
            linestyle="--", marker="s", markersize=5, zorder=4)

for cohorte in ultimas_n:
    mobs, vals = extraer_vel(cohorte)
    if not mobs:
        continue
    color = COLOR_2024 if cohorte.startswith("2024") else COLOR_2025
    ax.annotate(cohorte, xy=(mobs[-1], vals[-1]),
                xytext=(6, 0), textcoords="offset points",
                fontsize=9, fontweight="bold", color=color, va="center", zorder=5)
    ax.plot(mobs[-1], vals[-1], "o", color=color, markersize=6, zorder=5)

ax.text(0.98, 0.95, "MORA CRECE", transform=ax.transAxes, fontsize=9,
        ha="right", va="top", color="#d62728", alpha=0.5, fontstyle="italic")
ax.text(0.98, 0.05, "MORA MEJORA", transform=ax.transAxes, fontsize=9,
        ha="right", va="bottom", color="#2ca02c", alpha=0.5, fontstyle="italic")

ax.set_xlabel("MOB destino", fontsize=12)
ax.set_ylabel("Velocidad (cambio absoluto del índice)", fontsize=12)
ax.set_title("Velocidad de Mora — 1ra Derivada del Índice de Morosidad",
             fontsize=14, fontweight="bold")
mob_max = max(int(c.split("->")[1]) for c in velocidad.columns)
ax.set_xticks(range(2, mob_max + 1))
ax.grid(True, alpha=0.25, linestyle="--")
ax.set_xlim(1.5, mob_max + 1.5)

legend_elements = [
    Line2D([0], [0], color=COLOR_2024, alpha=ALPHA_IND, lw=4, label="Cohortes 2024"),
    Line2D([0], [0], color=COLOR_2024, lw=2.8, ls="--", marker="s", markersize=5, label="Promedio 2024"),
    Line2D([0], [0], color=COLOR_2025, alpha=ALPHA_IND, lw=4, label="Cohortes 2025"),
]
if pv_m25:
    legend_elements.append(
        Line2D([0], [0], color=COLOR_2025, lw=2.8, ls="--", marker="s", markersize=5, label="Promedio 2025"))
legend_elements.append(
    Line2D([0], [0], color="#d62728", lw=1.5, ls=":", label="Plateau (velocidad=0)"))
ax.legend(handles=legend_elements, loc="upper right", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "reports", "velocidad_mora.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado en reports/velocidad_mora.png")

### Diagnóstico de fase por cohorte (Velocidad + Factor combinados)

Panel izquierdo: **Velocidad** (1ra derivada, cambio absoluto).
Panel derecho: **Factor** (2da derivada, cambio relativo).

Juntos permiten identificar en qué fase del ciclo de vida está cada cosecha.


In [ ]:
# Panel combinado: Velocidad (izq) + Factor (der) con diagnóstico de fase
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), sharey=False)

# --- Recargar factores ---
factores = pd.read_csv(os.path.join(PROCESSED_DIR, "factores_desarrollo.csv"), sep=";", decimal=",", index_col=0)

cohortes_ord = sorted(velocidad.index)
ultimas_n = cohortes_ord[-3:]
cohortes_2024 = [c for c in cohortes_ord if c.startswith("2024")]
cohortes_2025 = [c for c in cohortes_ord if c.startswith("2025")]

# Promedios velocidad
def prom(df, cohortes):
    mo, vo = [], []
    for col in df.columns:
        v = df.loc[cohortes, col].dropna()
        if len(v) >= 2:
            mo.append(int(col.split("->")[1]))
            vo.append(v.mean())
    return mo, vo

pv_m24, pv_v24 = prom(velocidad, cohortes_2024)
pf_m24, pf_v24 = prom(factores, cohortes_2024)
pv_m25, pv_v25 = prom(velocidad, cohortes_2025) if len(cohortes_2025) >= 2 else ([], [])
pf_m25, pf_v25 = prom(factores, cohortes_2025) if len(cohortes_2025) >= 2 else ([], [])

# ---- PANEL IZQUIERDO: Velocidad ----
ax1.axhline(y=0, color="#d62728", linewidth=1.5, linestyle=":", alpha=0.7)
ax1.axhspan(0, 0.06, alpha=0.04, color="red")
ax1.axhspan(-0.03, 0, alpha=0.04, color="green")

for cohorte in cohortes_2024:
    v = velocidad.loc[cohorte].dropna()
    if len(v) == 0:
        continue
    mobs = [int(t.split("->")[1]) for t in v.index]
    hl = cohorte in ultimas_n
    ax1.plot(mobs, v.values, color=COLOR_2024,
             alpha=0.85 if hl else 0.15, linewidth=2.0 if hl else 0.8)

for cohorte in cohortes_2025:
    v = velocidad.loc[cohorte].dropna()
    if len(v) == 0:
        continue
    mobs = [int(t.split("->")[1]) for t in v.index]
    hl = cohorte in ultimas_n
    ax1.plot(mobs, v.values, color=COLOR_2025,
             alpha=0.85 if hl else 0.15, linewidth=2.0 if hl else 0.8)

if pv_m24:
    ax1.plot(pv_m24, pv_v24, color=COLOR_2024, lw=2.8, ls="--", marker="s", ms=4, label="Prom. 2024")
if pv_m25:
    ax1.plot(pv_m25, pv_v25, color=COLOR_2025, lw=2.8, ls="--", marker="s", ms=4, label="Prom. 2025")

ax1.set_title("Velocidad (1ra derivada)", fontsize=13, fontweight="bold")
ax1.set_xlabel("MOB destino", fontsize=11)
ax1.set_ylabel("\u0394 índice (cambio absoluto)", fontsize=11)
ax1.legend(fontsize=9, loc="upper right")
ax1.grid(True, alpha=0.25, linestyle="--")
mob_max = max(int(c.split("->")[1]) for c in velocidad.columns)
ax1.set_xticks(range(2, mob_max + 1))
ax1.text(0.02, 0.97, "CRECE", transform=ax1.transAxes, fontsize=8, color="#d62728", alpha=0.6, va="top")
ax1.text(0.02, 0.03, "MEJORA", transform=ax1.transAxes, fontsize=8, color="#2ca02c", alpha=0.6, va="bottom")

# ---- PANEL DERECHO: Factor ----
ax2.axhline(y=1.0, color="#d62728", linewidth=1.5, linestyle=":", alpha=0.7)
ax2.axhspan(1.0, 1.35, alpha=0.04, color="red")
ax2.axhspan(0.85, 1.0, alpha=0.04, color="green")

for cohorte in cohortes_2024:
    v = factores.loc[cohorte].dropna()
    if len(v) == 0:
        continue
    mobs = [int(t.split("->")[1]) for t in v.index]
    hl = cohorte in ultimas_n
    ax2.plot(mobs, v.values, color=COLOR_2024,
             alpha=0.85 if hl else 0.15, linewidth=2.0 if hl else 0.8)

for cohorte in cohortes_2025:
    v = factores.loc[cohorte].dropna()
    if len(v) == 0:
        continue
    mobs = [int(t.split("->")[1]) for t in v.index]
    hl = cohorte in ultimas_n
    ax2.plot(mobs, v.values, color=COLOR_2025,
             alpha=0.85 if hl else 0.15, linewidth=2.0 if hl else 0.8)

if pf_m24:
    ax2.plot(pf_m24, pf_v24, color=COLOR_2024, lw=2.8, ls="--", marker="s", ms=4, label="Prom. 2024")
if pf_m25:
    ax2.plot(pf_m25, pf_v25, color=COLOR_2025, lw=2.8, ls="--", marker="s", ms=4, label="Prom. 2025")

ax2.set_title("Factor de Desarrollo (2da derivada)", fontsize=13, fontweight="bold")
ax2.set_xlabel("MOB destino", fontsize=11)
ax2.set_ylabel("Factor (cambio relativo)", fontsize=11)
ax2.legend(fontsize=9, loc="upper right")
ax2.grid(True, alpha=0.25, linestyle="--")
ax2.set_xticks(range(2, mob_max + 1))
ax2.text(0.02, 0.97, "ACELERA", transform=ax2.transAxes, fontsize=8, color="#d62728", alpha=0.6, va="top")
ax2.text(0.02, 0.03, "DESACELERA", transform=ax2.transAxes, fontsize=8, color="#2ca02c", alpha=0.6, va="bottom")

fig.suptitle("Diagnóstico de Fase de Cosechas: Velocidad vs Factor",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "reports", "diagnostico_fases.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado en reports/diagnostico_fases.png")

In [ ]:
# Diagnóstico de fase para cada cohorte (basado en su última transición disponible)
print("DIAGNÓSTICO DE FASE POR COHORTE")
print("=" * 70)
print(f"{"Cohorte":>10s} | {"Última trans.":>13s} | {"Velocidad":>10s} | {"Factor":>8s} | Fase")
print("-" * 70)

for cohorte in sorted(velocidad.index):
    vel_vals = velocidad.loc[cohorte].dropna()
    fac_vals = factores.loc[cohorte].dropna()
    if len(vel_vals) == 0 or len(fac_vals) == 0:
        continue
    last_trans = vel_vals.index[-1]
    v = vel_vals.iloc[-1]
    f = fac_vals.iloc[-1]

    # Determinar fase
    if v > 0.005 and f > 1.02:
        fase = "DETERIORO (acelerando)"
    elif v > 0.005 and f <= 1.02:
        fase = "DESACELERACIÓN (cerca del plateau)"
    elif abs(v) <= 0.005:
        fase = "PLATEAU / MESETA"
    elif v < -0.005 and f < 0.98:
        fase = "CURACIÓN (mejorando)"
    else:
        fase = "ESTABILIZACIÓN"

    print(f"{cohorte:>10s} | {last_trans:>13s} | {v:>+10.4f} | {f:>8.4f} | {fase}")

print("=" * 70)


---
## 4. Índice de morosidad al mismo MOB (comparación entre cohortes)

Para comparar cohortes de forma justa, se mira el índice al mismo MOB.

In [ ]:
# MOB de referencia: el máximo MOB que tienen todas las cohortes
mob_comun = matriz.dropna(axis=1).columns
mob_max_comun = mob_comun[-1]
print(f"MOB máximo común a todas las cohortes: {mob_max_comun}\n")

comparacion = matriz[mob_comun].copy()
comparacion["último_indice"] = matriz.apply(lambda row: row.dropna().iloc[-1], axis=1)
comparacion["mob_actual"] = matriz.apply(lambda row: row.dropna().index[-1], axis=1)

print("Índice de morosidad por cohorte (MOBs comunes + último disponible):")
comparacion[[mob_comun[0], mob_comun[-1], "mob_actual", "último_indice"]].style.format(
    {mob_comun[0]: "{:.2%}", mob_comun[-1]: "{:.2%}", "último_indice": "{:.2%}"}
)

---
## 5. Evolución del monto moroso por cohorte

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for cohorte, grupo in df.groupby("cohorte"):
    ax.plot(grupo["mob"], grupo["moroso"] / 1e6, marker="o", markersize=4, linewidth=2, label=cohorte)

ax.set_xlabel("Months on Books (MOB)", fontsize=12)
ax.set_ylabel("Monto Moroso (millones)", fontsize=12)
ax.set_title("Evolución del Monto Moroso Acumulado por Cohorte", fontsize=14, fontweight="bold")
ax.legend(title="Cohorte", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, int(df['mob'].max()) + 1))
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "reports", "monto_moroso.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado en reports/monto_moroso.png")

---
## 6. Resumen Ejecutivo

In [ ]:
print("=" * 60)
print("RESUMEN EJECUTIVO - ANÁLISIS VINTAGE")
print("=" * 60)

for cohorte, grupo in df.groupby("cohorte"):
    ultimo = grupo.iloc[-1]
    print(f"\nCohorte {cohorte}:")
    print(f"  MOB actual:          {int(ultimo['mob'])}")
    print(f"  Índice actual:       {ultimo['indice']:.2%}")
    print(f"  Índice máximo:       {grupo['indice'].max():.2%} (MOB {int(grupo.loc[grupo['indice'].idxmax(), 'mob'])})")
    print(f"  Índice mínimo:       {grupo['indice'].min():.2%} (MOB {int(grupo.loc[grupo['indice'].idxmin(), 'mob'])})")
    print(f"  Total vencido:       ${ultimo['total_vencido']:,.0f}")
    print(f"  Monto moroso:        ${ultimo['moroso']:,.0f}")
    tendencia = "DESCENDENTE" if ultimo['indice'] < grupo['indice'].max() else "EN MÁXIMO"
    print(f"  Tendencia:           {tendencia}")

print("\n" + "=" * 60)